# Ultrasound Boundary Detection Pipeline - Example Usage

This notebook demonstrates how to use the traditional boundary detection pipeline for ultrasound videos.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('src')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Import pipeline components
from video_loader import VideoLoader
from preprocessing import UltrasoundPreprocessor
from roi_detector import ROIDetector
from boundary_detectors import (
    create_canny_detector,
    create_active_contour_detector,
    create_watershed_detector
)
from boundary_refiner import BoundaryRefiner
from temporal_tracker import TemporalTracker
from visualizer import BoundaryVisualizer
from pipeline import UltrasoundBoundaryPipeline

# Setup plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

## 2. Quick Start - Full Pipeline

In [ ]:
# Path to your ultrasound video
VIDEO_PATH = 'data/input/sample_ultrasound.mp4'

# Create pipeline with default configuration
pipeline = UltrasoundBoundaryPipeline('config/params.yaml')

# Process first 50 frames for quick testing
pipeline.config['processing']['max_frames'] = 50

# Run pipeline
results = pipeline.process_video(VIDEO_PATH)

print(f"Processed {len(results)} frames")
print(f"Results saved to: {pipeline.config['output']['directory']}")

## 3. Step-by-Step Usage - Individual Components

### 3.1 Load Video and Extract Frame

In [ ]:
# Load video
loader = VideoLoader(VIDEO_PATH)

# Print metadata
print("Video Metadata:")
for key, value in loader.get_metadata().items():
    print(f"  {key}: {value}")

# Extract first frame
frame = loader.extract_frame(0)

# Display
plt.figure(figsize=(10, 6))
plt.imshow(frame, cmap='gray')
plt.title('Original Frame')
plt.axis('off')
plt.show()

### 3.2 Preprocessing

In [ ]:
# Create preprocessor
preprocessor = UltrasoundPreprocessor()

# Preprocess with intermediate results
results = preprocessor.preprocess(frame, return_intermediate=True)

# Display preprocessing stages
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

stages = ['original', 'denoised', 'enhanced', 'final']
for idx, stage in enumerate(stages):
    if stage in results:
        axes[idx].imshow(results[stage], cmap='gray')
        axes[idx].set_title(f'{stage.capitalize()}')
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 3.3 ROI Detection

In [ ]:
# Detect ROI
roi_detector = ROIDetector()
roi, bbox, debug_info = roi_detector.detect_roi(results['final'], return_debug_info=True)

print(f"ROI bounding box: {bbox}")
print(f"ROI shape: {roi.shape}")
print(f"Number of components found: {debug_info['num_components']}")

# Visualize ROI
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original with ROI
vis = roi_detector.visualize_roi(results['final'], bbox)
axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[0].set_title('Detected ROI')
axes[0].axis('off')

# Binary mask
axes[1].imshow(debug_info['binary'], cmap='gray')
axes[1].set_title('Binary Threshold')
axes[1].axis('off')

# ROI crop
axes[2].imshow(roi, cmap='gray')
axes[2].set_title('Extracted ROI')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 3.4 Boundary Detection with Multiple Methods

In [ ]:
# Create detectors
canny_det = create_canny_detector()
active_contour_det = create_active_contour_detector()
watershed_det = create_watershed_detector()

# Detect with each method
boundaries = {}

# Canny
print("Running Canny detection...")
boundaries['canny'] = canny_det.detect_combined(roi)

# Active contour
print("Running Active Contour detection...")
ac_contour = active_contour_det.detect_with_edge_initialization(roi, boundaries['canny'])
boundaries['active_contour'] = active_contour_det.contour_to_mask(ac_contour, roi.shape)

# Watershed
print("Running Watershed detection...")
labels, vessel_mask, _ = watershed_det.detect_full_pipeline(roi)
boundaries['watershed'] = vessel_mask

# Display all detections
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

axes[0].imshow(roi, cmap='gray')
axes[0].set_title('Original ROI')
axes[0].axis('off')

for idx, (name, boundary) in enumerate(boundaries.items(), 1):
    axes[idx].imshow(boundary, cmap='gray')
    axes[idx].set_title(f'{name.replace("_", " ").title()}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 3.5 Boundary Refinement and Fusion

In [ ]:
# Create refiner
refiner = BoundaryRefiner()

# Fuse and refine boundaries
refined, metrics = refiner.ensemble_refine(boundaries, roi)

print("Boundary Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.3f}")

# Display fusion results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Voting fusion
fused_voting = refiner.fuse_boundaries(boundaries)
axes[0].imshow(fused_voting, cmap='gray')
axes[0].set_title('Voting Fusion')
axes[0].axis('off')

# Weighted fusion
refiner.config['fusion_method'] = 'weighted_average'
fused_weighted = refiner.fuse_boundaries(boundaries)
axes[1].imshow(fused_weighted, cmap='gray')
axes[1].set_title('Weighted Fusion')
axes[1].axis('off')

# Final refined
axes[2].imshow(refined, cmap='gray')
axes[2].set_title('Refined Boundary')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 3.6 Overlay Boundary on Original Frame

In [ ]:
# Extract contour from refined boundary
contours = refiner._extract_contours(refined)
if len(contours) > 0:
    final_contour = refiner._select_best_contour(contours)
    
    # Create visualization
    vis_frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    
    # Offset contour by ROI position
    x, y, w, h = bbox
    contour_offset = final_contour.copy()
    contour_offset[:, :, 0] += x
    contour_offset[:, :, 1] += y
    
    # Draw ROI box
    cv2.rectangle(vis_frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
    
    # Draw boundary
    cv2.drawContours(vis_frame, [contour_offset], 0, (0, 255, 0), 2)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
    plt.title('Detected Boundary Overlay')
    plt.axis('off')
    plt.show()
else:
    print("No contours found!")

## 4. Temporal Tracking Demo

In [ ]:
# Process multiple frames with tracking
tracker = TemporalTracker()
num_frames_to_track = 30

tracking_results = []

for i in range(num_frames_to_track):
    # Load frame
    frame_i = loader.extract_frame(i)
    if frame_i is None:
        break
    
    # Preprocess
    preprocessed_i = preprocessor.preprocess(frame_i)
    
    # Detect ROI
    roi_i, bbox_i = roi_detector.detect_roi(preprocessed_i)
    
    # Detect boundary (using Canny for speed)
    edges_i = canny_det.detect_combined(roi_i)
    contour_i = canny_det.get_dominant_boundary(edges_i)
    
    # Track
    if contour_i is not None:
        tracked_contour, tracked_bbox, confidence = tracker.track(roi_i, contour_i, bbox_i)
        
        # Store results
        area = cv2.contourArea(tracked_contour) if tracked_contour is not None else 0
        tracking_results.append({
            'frame': i,
            'area': area,
            'confidence': confidence
        })

print(f"Tracked {len(tracking_results)} frames")

In [ ]:
# Plot tracking results
if tracking_results:
    frames = [r['frame'] for r in tracking_results]
    areas = [r['area'] for r in tracking_results]
    confidences = [r['confidence'] for r in tracking_results]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))
    
    # Area over time
    ax1.plot(frames, areas, 'b-', linewidth=2)
    ax1.set_xlabel('Frame Number')
    ax1.set_ylabel('Vessel Area (pixels)')
    ax1.set_title('Vessel Area Over Time')
    ax1.grid(True, alpha=0.3)
    
    # Confidence over time
    ax2.plot(frames, confidences, 'g-', linewidth=2)
    ax2.set_xlabel('Frame Number')
    ax2.set_ylabel('Tracking Confidence')
    ax2.set_title('Tracking Confidence Over Time')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1.1])
    
    plt.tight_layout()
    plt.show()
    
    # Compression event detection
    event_info = tracker.detect_compression_event()
    print("\nCompression Event Analysis:")
    print(f"  Compression detected: {event_info.get('compression_detected', False)}")
    print(f"  Release detected: {event_info.get('release_detected', False)}")
    print(f"  Area change ratio: {event_info.get('area_change_ratio', 0):.3f}")
    print(f"  Compression ratio: {event_info.get('compression_ratio', 0):.3f}")

## 5. Benchmark Different Detectors

In [ ]:
import time

# Benchmark on preprocessed ROI
timings = {}

# Canny
start = time.time()
_ = canny_det.detect_combined(roi)
timings['Canny'] = time.time() - start

# Active Contour
start = time.time()
_ = active_contour_det.detect(roi)
timings['Active Contour'] = time.time() - start

# Watershed
start = time.time()
_ = watershed_det.detect_full_pipeline(roi)
timings['Watershed'] = time.time() - start

# Plot benchmark results
fig, ax = plt.subplots(figsize=(10, 6))
methods = list(timings.keys())
times = list(timings.values())

bars = ax.bar(methods, times, color=['blue', 'green', 'orange'])
ax.set_ylabel('Time (seconds)')
ax.set_title('Detector Performance Benchmark')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}s',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nBenchmark Results:")
for method, t in timings.items():
    print(f"  {method:20s}: {t:.3f} seconds")

## 6. Parameter Tuning Examples

In [ ]:
# Example: Compare different CLAHE settings
clip_limits = [1.0, 2.0, 3.0, 4.0]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, clip_limit in enumerate(clip_limits):
    # Create preprocessor with specific CLAHE setting
    config = {'clahe_clip_limit': clip_limit}
    prep = UltrasoundPreprocessor(config)
    
    # Preprocess
    enhanced = prep.preprocess(frame)
    
    # Display
    axes[idx].imshow(enhanced, cmap='gray')
    axes[idx].set_title(f'CLAHE Clip Limit = {clip_limit}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Example: Compare different active contour parameters
alpha_values = [0.005, 0.015, 0.05, 0.1]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, alpha in enumerate(alpha_values):
    config = {'alpha': alpha}
    ac_det = create_active_contour_detector(config)
    
    contour = ac_det.detect(roi)
    
    # Visualize
    vis = roi.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    # Draw contour
    contour_int = np.round(contour).astype(np.int32)
    cv2.polylines(vis, [contour_int], True, (0, 255, 0), 2)
    
    axes[idx].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f'Alpha (elasticity) = {alpha}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Cleanup

In [ ]:
# Release video resources
loader.release()

print("Done!")

## Summary

This notebook demonstrated:

1. **Quick start** with the full pipeline
2. **Step-by-step usage** of individual components
3. **Multiple detection methods** and their comparison
4. **Boundary refinement** and fusion techniques
5. **Temporal tracking** across video frames
6. **Performance benchmarking** of different detectors
7. **Parameter tuning** examples

For more information, see the README.md file and configuration documentation.